In [27]:
from pyspark.sql import SparkSession 

from pyspark.sql.functions import *

from pyspark.sql.types import*

spark=SparkSession.builder.appName("Spark SQL day2").master("local[*]").getOrCreate()

ConnectionRefusedError: [Errno 61] Connection refused

| emp_id | name  | salary |
|--------|-------|--------|
| 1      | Alice | 90000  |
| 2      | Bob   | 75000  |
| 3      | Carol | 90000  |
| 4      | Dave  | 120000 |
| 5      | Eve   | 60000  |


find 2nd higest salary without window function 






In [3]:




# Sample data
data = [
    (1, "Alice", 90000),
    (2, "Bob", 75000),
    (3, "Carol", 90000),
    (4, "Dave", 120000),
    (5, "Eve", 60000)
]

# Column names
columns = ["emp_id", "name", "salary"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Show DataFrame
df.show()

+------+-----+------+
|emp_id| name|salary|
+------+-----+------+
|     1|Alice| 90000|
|     2|  Bob| 75000|
|     3|Carol| 90000|
|     4| Dave|120000|
|     5|  Eve| 60000|
+------+-----+------+



In [4]:
df.createOrReplaceTempView("employees")



In [7]:
spark.sql("""
             select max(salary)
             from employees
             where  salary < (select max(salary) from employees) 
          
          """).show()

+-----------+
|max(salary)|
+-----------+
|      90000|
+-----------+



In [ ]:
# another way
spark.sql("""
             select salary 
             from employees
             order by salary desc
             limit 1 offset 1
          
          """).show()

+------+
|salary|
+------+
| 75000|
+------+



In [ ]:
# nth higest salary 

spark.sql("""
             select salary 
             from employees
             order by salary desc
             limit 1 offset n-1
          
          """).show()

# replace n with desired values



COUNT(*) vs COUNT(col) — understanding NULLs

| order_id | cust_id | discount | status     |
|----------|---------|-----------|------------|
| 1        | 101     | NULL      | shipped    |
| 2        | 102     | 10        | shipped    |
| 3        | 101     | NULL      | pending    |
| 4        | 103     | 5         | shipped    |
| 5        | 102     | NULL      | cancelled  |

In [11]:
# Sample data
data = [
    (1, 101, None, "shipped"),
    (2, 102, 10, "shipped"),
    (3, 101, None, "pending"),
    (4, 103, 5, "shipped"),
    (5, 102, None, "cancelled")
]

# Column names
columns = ["order_id", "cust_id", "discount", "status"]

order_df=spark.createDataFrame(data,columns)

In [12]:
order_df.createOrReplaceTempView("orders")

In [14]:
spark.sql("""SELECT
  COUNT(*)                          AS total_rows,
  COUNT(discount)                   AS rows_with_discount,
  COUNT(*) - COUNT(discount)       AS null_discounts,
  COUNT(DISTINCT cust_id)           AS unique_customers,
  COUNT(CASE WHEN status = 'shipped' THEN 1 END) AS shipped_orders
FROM orders
"""
).show()

+----------+------------------+--------------+----------------+--------------+
|total_rows|rows_with_discount|null_discounts|unique_customers|shipped_orders|
+----------+------------------+--------------+----------------+--------------+
|         5|                 2|             3|               3|             3|
+----------+------------------+--------------+----------------+--------------+



### Key insight
    COUNT(*) counts all rows including NULLs.
     COUNT(col) skips NULLs. This distinction is asked in nearly every service company SQL round. 
     
     COUNT(DISTINCT col) counts unique non-NULL values.

####  CROSS JOIN to generate all combinations

size table:
----------
| size |
|------|
| S    |
| M    |
| L    |

colors table:
| color |
|-------|
| Red   |
| Blue  |
| Green |


In [22]:
# to generate all combination we can use cross join 

# Generate all size × color combinations (useful for inventory gaps)

size_data =[
    ("s",),("M",),("L",)
]
size_df=spark.createDataFrame(data,["size"])

color_data=[
    ("Red",),
    ("Blue",),
    ("Green",)
]

color_df = spark.createDataFrame(color_data, ["color"])

color_df.show()

+-----+
|color|
+-----+
|  Red|
| Blue|
|Green|
+-----+



In [23]:
color_df.createOrReplaceTempView("color")

size_df.createOrReplaceTempView("size")

In [ ]:
# i wanted to check all possible combination 

spark.sql("""
             SELECT s.size, c.color
             FROM  size  s
              CROSS JOIN color c
             ORDER BY s.size, c.color
              
          
          """).show()

+----+-----+
|size|color|
+----+-----+
|   1| Blue|
|   1|Green|
|   1|  Red|
|   2| Blue|
|   2|Green|
|   2|  Red|
|   3| Blue|
|   3|Green|
|   3|  Red|
|   4| Blue|
|   4|Green|
|   4|  Red|
|   5| Blue|
|   5|Green|
|   5|  Red|
+----+-----+



26/05/08 16:31:53 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1048407 ms exceeds timeout 120000 ms
26/05/08 16:31:53 WARN SparkContext: Killing executors is not supported by current scheduler.
26/05/08 16:32:02 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1363)
	at 